# 3.2 — Semantic search over IT docs
**Session 3.2 · Demo · retrieval layer**

What you'll see: the help-desk documents turned into vectors, stored in Chroma, and searched
three ways. Keyword search, vector search and a hybrid each win on different questions.

In [1]:
# @title Setup: install the search tools and load the embedding model { display-mode: "form" }
# No API key is needed in this notebook.
KIT_URL = "git+https://github.com/fastbatchai/helpdesk-assistant.git"
KIT = f"helpdesk-kit[rag] @ {KIT_URL}"

import importlib, importlib.util, shutil, subprocess, sys
# uv comes with Colab and installs much faster than pip; pip is the fallback.
INSTALL = (["uv", "pip", "install", "--python", sys.executable, "-q"] if shutil.which("uv")
           else [sys.executable, "-m", "pip", "install", "-q"])
if any(importlib.util.find_spec(module) is None for module in ("helpdesk_kit", "chromadb")):
    subprocess.check_call([*INSTALL, KIT])
    importlib.invalidate_caches()

import html
import re

import numpy as np
from IPython.display import HTML, display
from sentence_transformers import SentenceTransformer

from helpdesk_kit.retrieval import EMBEDDING_MODEL, Chunk, Retriever, load_kb

embedder = SentenceTransformer(EMBEDDING_MODEL)

# Display helpers, so the results read well on a shared screen.
INK, SOFT_INK, BLUE, ORANGE, AQUA, GREY, GRID = (
    "#0b0b0b", "#52514e", "#2a78d6", "#eb6834", "#1baf7a", "#898781", "#e1e0d9"
)
METHOD_COLOURS = {"keyword": ORANGE, "vector": BLUE, "hybrid": AQUA}


def badge(text, colour):
    """A small coloured label."""
    return (
        f'<span style="display:inline-block;padding:1px 8px;margin-right:4px;border-radius:10px;'
        f'background:{colour};color:#fff;font-size:12px;font-weight:600">{html.escape(text)}</span>'
    )


def card(body, caption="", border=BLUE):
    """A white box with a coloured left edge, and an optional grey caption underneath."""
    caption_html = f'<div style="margin-top:6px;font-size:12px;color:{SOFT_INK}">{caption}</div>' if caption else ""
    return (
        f'<div style="border-left:3px solid {border};background:#fff;padding:8px 12px;margin:6px 0;'
        f'color:{INK}">{body}{caption_html}</div>'
    )


def headline(text):
    display(HTML(f'<div style="font-size:16px;font-weight:600;color:{INK};margin:4px 0 8px">{text}</div>'))


def highlight(text, query):
    """The text, with the words of the query in bold blue."""
    words = [word for word in re.findall(r"[\w-]+", query) if len(word) > 2]
    escaped = html.escape(text)
    if not words:
        return escaped
    pattern = re.compile(r"\b(" + "|".join(re.escape(html.escape(word)) for word in words) + r")\b", re.IGNORECASE)
    return pattern.sub(lambda match: f'<b style="color:{BLUE}">{match.group(0)}</b>', escaped)


def snippet(text, query, length=30):
    """About `length` words of the chunk, starting just before the first query word it contains."""
    words = text.split()
    query_words = {word.lower() for word in re.findall(r"[\w-]+", query) if len(word) > 2}
    start = next(
        (i for i, word in enumerate(words) if re.sub(r"[^\w-]", "", word).lower() in query_words), 0
    )
    start = max(start - 5, 0)
    piece = " ".join(words[start : start + length])
    return ("… " if start else "") + piece + (" …" if start + length < len(words) else "")


def hit_card(rank, chunk, query, border=BLUE):
    """One search result: rank, document title, a snippet with the query words highlighted."""
    hr_badge = badge("HR only", ORANGE) if chunk.audience != "all" else ""
    body = (
        f'<div style="font-weight:600">{rank}. {html.escape(chunk.title)} {hr_badge}</div>'
        f'<div style="font-size:13.5px;margin-top:4px">{highlight(snippet(chunk.text, query), query)}</div>'
    )
    return card(body, html.escape(chunk.id), border)


def show_columns(columns):
    """Side-by-side columns: a coloured title, then the HTML underneath."""
    cells = "".join(
        f'<td style="vertical-align:top;padding:0 8px;width:{100 // len(columns)}%">'
        f'<div style="font-weight:600;color:{METHOD_COLOURS.get(name, INK)};margin-bottom:4px">{html.escape(name)}</div>'
        f"{content}</td>"
        for name, content in columns.items()
    )
    display(HTML(f'<table style="border-collapse:collapse;width:100%"><tr>{cells}</tr></table>'))


def model_label():
    """The embedding model's name, shown wherever we turn text into numbers."""
    size = getattr(embedder, "get_embedding_dimension", embedder.get_sentence_embedding_dimension)()
    return (f'<span style="display:inline-block;margin:2px 0 6px;padding:3px 10px;border-radius:12px;'
            f'background:#eef3fb;border:1px solid {BLUE};color:{INK};font-size:12.5px">Embedding model: '
            f'<a href="https://huggingface.co/{EMBEDDING_MODEL}" target="_blank" style="color:{BLUE};'
            f'font-weight:600;text-decoration:none">{EMBEDDING_MODEL}</a> · {size} numbers per text</span>')


display(HTML(f"Ready. {model_label()}"))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 1. Chunking: five ways to cut a document
Search doesn't return whole documents: it returns **chunks**, the pieces each document is cut into. How we
cut changes what search can find, and what the model reads.

In [2]:
# @title Pick a document { display-mode: "form" }
DOCUMENT = "vpn-troubleshooting"  # @param ["account-lockout", "admin-rights-policy", "approved-software-list", "backup-and-restore", "data-classification", "email-and-calendar", "guest-wifi", "help-desk-policy", "incident-severity-levels", "it-equipment-expenses", "laptop-replacement-policy", "mfa-enrolment", "mobile-devices", "offboarding", "on-call-escalation", "onboarding", "password-policy-2021", "password-reset-policy", "phishing-reporting", "printer-setup", "salary-bands-hr", "shared-drives", "software-request-process", "teams-meetings", "travel-and-roaming", "vpn-setup", "vpn-troubleshooting"]

docs = load_kb()
doc = next(d for d in docs if d.id == DOCUMENT)
words = doc.text.split()
sections = [line for line in doc.text.splitlines() if line.startswith("#")]
display(HTML(card(
    f'<div style="font-weight:600;font-size:15px">{html.escape(doc.title)}</div>'
    f'<div style="color:{SOFT_INK};margin:4px 0">{len(words)} words · {len(sections)} headings · '
    f'updated {doc.updated}</div>'
    f'<details><summary style="color:{BLUE};cursor:pointer">read the document</summary>'
    f'<div style="white-space:pre-wrap;font-size:13px;margin-top:6px">{html.escape(doc.text)}</div></details>'
)))

### Cut it five ways
Each card shows the document coloured by chunk: the chunks alternate blue and orange, each start is
numbered, and words that belong to two chunks (the overlap) are purple. The two sliders set the size and
the overlap for the first two methods.

In [3]:
# @title Cut it five ways { display-mode: "form" }
CHUNK_WORDS = 120  # @param {type:"slider", min:40, max:200, step:10}
OVERLAP_WORDS = 20  # @param {type:"slider", min:0, max:60, step:5}


def lines_with_positions(doc):
    """Each line of the document, with the positions of its first and last words."""
    position, lines = 0, []
    for line in doc.text.splitlines():
        count = len(line.split())
        if count:
            lines.append((line, position, position + count))
        position += count
    return lines


def fixed_size(doc, size, overlap=0):
    """Windows of `size` words; each one starts `size - overlap` words after the one before."""
    total = len(doc.text.split())
    return [(start, min(start + size, total)) for start in range(0, max(total - overlap, 1), size - overlap)]


def structure_based(doc):
    """One chunk per section: a new chunk starts at every heading."""
    starts = [start for line, start, _ in lines_with_positions(doc) if line.startswith("#")]
    starts = sorted({0, *starts})
    return list(zip(starts, starts[1:] + [len(doc.text.split())]))


def sentences(doc):
    """The document as sentences (a heading or a list item counts as one), with their positions."""
    result = []
    for line, start, _ in lines_with_positions(doc):
        for piece in re.split(r"(?<=[^\d\s][.!?])\s+", line):  # "1." in a list doesn't end a sentence.
            count = len(piece.split())
            if count:
                result.append((piece, start, start + count))
                start += count
    return result


def semantic(doc, smallest=25, largest=200):
    """Grow each chunk sentence by sentence. A sentence joins the chunk if its meaning is close to the
    chunk so far (the average of the chunk's sentences, as numbers); if not, it starts a new chunk.
    "Close" is measured against this document: a sentence starts a new chunk when its similarity is
    in the lowest quarter of the similarities between neighbouring sentences. A chunk has at least
    `smallest` words and at most `largest`."""
    parts = sentences(doc)
    total = len(doc.text.split())
    if len(parts) < 2:
        return [(0, total)]
    vectors = embedder.encode([text for text, _, _ in parts], normalize_embeddings=True)
    neighbours = [float(vectors[i] @ vectors[i + 1]) for i in range(len(parts) - 1)]
    close_enough = np.percentile(neighbours, 25)

    starts, chunk_vectors = [0], [vectors[0]]
    for i in range(1, len(parts)):
        chunk_so_far = np.mean(chunk_vectors, axis=0)
        similarity = float(chunk_so_far @ vectors[i] / np.linalg.norm(chunk_so_far))
        start = parts[i][1]
        words_so_far = start - starts[-1]
        too_different = similarity < close_enough and words_so_far >= smallest
        if (too_different or words_so_far + (parts[i][2] - start) > largest) and total - start >= smallest:
            starts.append(start)
            chunk_vectors = [vectors[i]]
        else:
            chunk_vectors.append(vectors[i])
    return list(zip(starts, starts[1:] + [total]))


def context_line(doc, start):
    """Where a chunk comes from: the document, and the section it starts in."""
    heading = doc.title
    for line, line_start, _ in lines_with_positions(doc):
        if line_start > start:
            break
        if line.startswith("## "):
            heading = line.lstrip("# ").strip()
    return f"{doc.title} › {heading}: " if heading != doc.title else f"{doc.title}: "


METHODS = {
    "fixed size": lambda doc: fixed_size(doc, CHUNK_WORDS),
    "fixed size with overlap": lambda doc: fixed_size(doc, CHUNK_WORDS, OVERLAP_WORDS),
    "structure-based": structure_based,
    "semantic": semantic,
    "contextual": structure_based,  # The same cuts, plus a line saying where each chunk comes from.
}
HOW = {
    "fixed size": f"Every {CHUNK_WORDS} words, wherever they fall. Simple, but it cuts sentences and sections in half.",
    "fixed size with overlap": f"The same windows, but each repeats the last {OVERLAP_WORDS} words of the one "
                               "before (purple), so a sentence cut at the edge is still whole in one chunk.",
    "structure-based": "One chunk per section, following the document's headings. Chunks match the author's "
                       "topics, but their sizes vary a lot.",
    "semantic": "Grows each chunk sentence by sentence: the next sentence joins if its meaning is close to the "
                "chunk so far, otherwise it starts a new chunk (25 to 200 words). The embedding model measures "
                "how close.",
    "contextual": "The structure-based chunks, each starting with an added line saying where it comes from "
                  "(green, not in the original document). The full technique asks an LLM to write that line; here "
                  "we build it from the headings.",
}


def cut(doc, method):
    """The chunks of one document: (start, end, context line, text)."""
    words = doc.text.split()
    return [
        (start, end, context_line(doc, start) if method == "contextual" else "",
         (context_line(doc, start) if method == "contextual" else "") + " ".join(words[start:end]))
        for start, end in METHODS[method](doc)
    ]


TINTS, OVERLAP_TINT = ["#dbe8f8", "#fbe1d6"], "#e3d4f5"


def chunk_map(doc, pieces):
    """The document, each word coloured by the chunk it belongs to; a number marks each chunk's start,
    followed by the chunk's added context line in green, if it has one."""
    words = doc.text.split()
    owners = [[] for _ in words]
    for number, (start, end, _, _) in enumerate(pieces):
        for position in range(start, end):
            owners[position].append(number)
    starts, contexts = {}, {}
    for number, (start, _, context, _) in enumerate(pieces):
        starts.setdefault(start, number)
        contexts.setdefault(start, context)
    html_words = []
    for position, word in enumerate(words):
        if position in starts:
            html_words.append(f'<span style="display:inline-block;min-width:16px;padding:0 4px;border-radius:8px;'
                              f'background:{INK};color:#fff;font-size:10px;text-align:center">{starts[position] + 1}</span>')
            if contexts[position]:
                html_words.append(f'<span style="background:#d5f0e5;font-weight:600;border-radius:3px;padding:0 3px">'
                                  f'{html.escape(contexts[position].strip())}</span>')
        tint = OVERLAP_TINT if len(owners[position]) > 1 else TINTS[owners[position][0] % 2] if owners[position] else "#fff"
        html_words.append(f'<span style="background:{tint}">{html.escape(word)}</span>')
    return f'<div style="line-height:1.9;font-size:12.5px">{" ".join(html_words)}</div>'


chunked = {method: cut(doc, method) for method in METHODS}
for method, pieces in chunked.items():
    sizes = [len(text.split()) for _, _, _, text in pieces]
    extra = f"<div>{model_label()}</div>" if method == "semantic" else ""
    display(HTML(card(
        f'<div style="font-weight:600;font-size:15px">{html.escape(method)} · {len(pieces)} chunks</div>'
        f'<div style="color:{SOFT_INK};margin:2px 0 6px">{html.escape(HOW[method])}</div>{extra}'
        + chunk_map(doc, pieces),
        f"chunk size in words: smallest {min(sizes)}, average {np.mean(sizes):.0f}, largest {max(sizes)}",
    )))

### Two chunks, method by method
For each method: the chunk that contains a phrase, and the chunk after it. **Watch for:** where each
method cuts, and the line the contextual method adds.

In [4]:
# @title Two chunks, method by method { display-mode: "form" }
PHRASE = "NGL-117"  # @param {type:"string"}

rows = ""
for method, pieces in chunked.items():
    found = next((n for n, piece in enumerate(pieces) if PHRASE.lower() in piece[3].lower()), 0)
    cells = ""
    for number in (found, found + 1):
        if number >= len(pieces):
            cells += '<td style="width:42%"></td>'
            continue
        _, _, context, text = pieces[number]
        body = highlight(text[len(context):], PHRASE)
        if context:
            body = f'<span style="background:#d5f0e5;font-weight:600">{html.escape(context)}</span>' + body
        cells += (f'<td style="vertical-align:top;padding:6px 10px;width:42%;border-top:1px solid #e1e0d9">'
                  f'<div style="font-size:11px;color:{SOFT_INK};margin-bottom:3px">chunk {number + 1} of {len(pieces)}</div>'
                  f'<div style="font-size:12.5px">{body}</div></td>')
    rows += (f'<tr><td style="vertical-align:top;padding:6px 10px 6px 0;width:16%;font-weight:600;'
             f'border-top:1px solid #e1e0d9">{html.escape(method)}</td>{cells}</tr>')
display(HTML(f'<table style="border-collapse:collapse;width:100%;background:#fff">{rows}</table>'))

fixed size,"chunk 2 of 3answered within 60 seconds. Connect again and approve the request promptly. Check your phone has a data connection. **NGL-117: session dropped by client.** The connection was lost on your side. This commonly appears as repeated disconnects on Wi-Fi. Turn off the power-saving setting on your wireless adapter and update NorthLink. If the disconnects continue, open a ticket and attach the log from Help > Export Logs. **NGL-530: route conflict.** Another network, often a home router or a second VPN, is using the same address range. Disconnect any other VPN. If you are at home, restart your router. **NGL-221: account not permitted.** Your account is not in the VPN access group. This is common in a contractor's first week. Your manager","chunk 3 of 3must request access through the portal. ## Slow connections All traffic passes through the tunnel, so speed depends on your own connection. Disconnect from the VPN for video calls. When working abroad, also read the Travel and Roaming guide. ## When to contact us Open a ticket if a code above keeps returning or is not listed. Include the code, time and your location. If a whole office cannot connect, phone the help desk, because it is treated as a P1 incident."
fixed size with overlap,"chunk 2 of 4office network and restart, or contact the help desk for a renewal. **NGL-412: MFA timeout.** The approval request was not answered within 60 seconds. Connect again and approve the request promptly. Check your phone has a data connection. **NGL-117: session dropped by client.** The connection was lost on your side. This commonly appears as repeated disconnects on Wi-Fi. Turn off the power-saving setting on your wireless adapter and update NorthLink. If the disconnects continue, open a ticket and attach the log from Help > Export Logs. **NGL-530: route conflict.** Another network, often a home router or a second VPN, is using the same address range. Disconnect any other VPN. If you are at home, restart your router. **NGL-221: account not","chunk 3 of 4using the same address range. Disconnect any other VPN. If you are at home, restart your router. **NGL-221: account not permitted.** Your account is not in the VPN access group. This is common in a contractor's first week. Your manager must request access through the portal. ## Slow connections All traffic passes through the tunnel, so speed depends on your own connection. Disconnect from the VPN for video calls. When working abroad, also read the Travel and Roaming guide. ## When to contact us Open a ticket if a code above keeps returning or is not listed. Include the code, time and your location. If a whole office cannot connect, phone the help desk, because it is treated as a"
structure-based,"chunk 2 of 4## Error codes **NGL-809: certificate expired.** Your laptop's device certificate has expired, usually because the laptop has been away from the network for more than 60 days. You cannot fix this yourself. Connect to an office network and restart, or contact the help desk for a renewal. **NGL-412: MFA timeout.** The approval request was not answered within 60 seconds. Connect again and approve the request promptly. Check your phone has a data connection. **NGL-117: session dropped by client.** The connection was lost on your side. This commonly appears as repeated disconnects on Wi-Fi. Turn off the power-saving setting on your wireless adapter and update NorthLink. If the disconnects continue, open a ticket and attach the log from Help > Export Logs. **NGL-530: route conflict.** Another network, often a home router or a second VPN, is using the same address range. Disconnect any other VPN. If you are at home, restart your router. **NGL-221: account not permitted.** Your account is not in the VPN access group. This is common in a contractor's first week. Your manager must request access through the portal.","chunk 3 of 4## Slow connections All traffic passes through the tunnel, so speed depends on your own connection. Dis

### Cut all the documents
The five methods on all 27 documents. Pick the one the rest of this notebook uses.

In [5]:
# @title Cut all the documents { display-mode: "form" }
USE_METHOD = "fixed size with overlap"  # @param ["fixed size", "fixed size with overlap", "structure-based", "semantic", "contextual"]

all_chunks = {
    method: [
        Chunk(id=f"{d.id}#{n}", doc_id=d.id, title=d.title, department=d.department, updated=d.updated,
              audience=d.audience, text=text)
        for d in docs
        for n, (_, _, _, text) in enumerate(cut(d, method))
    ]
    for method in METHODS
}
rows = "".join(
    f'<tr style="{"font-weight:600;background:#eef3fb" if method == USE_METHOD else ""}">'
    f'<td style="padding:6px 14px 6px 4px">{html.escape(method)}</td>'
    + "".join(f'<td style="padding:6px 14px;text-align:right">{value}</td>' for value in (
        len(chunks_), min(len(c.text.split()) for c in chunks_),
        f"{np.mean([len(c.text.split()) for c in chunks_]):.0f}", max(len(c.text.split()) for c in chunks_)))
    + "</tr>"
    for method, chunks_ in all_chunks.items()
)
header = "".join(f'<th style="padding:6px 14px;text-align:right;color:{SOFT_INK}">{name}</th>'
                 for name in ("chunks", "smallest", "average", "largest"))
display(HTML(f'<table style="border-collapse:collapse;background:#fff"><tr><th style="text-align:left;'
             f'padding:6px 14px 6px 4px;color:{SOFT_INK}">method (sizes in words)</th>{header}</tr>{rows}</table>'))

chunks = all_chunks[USE_METHOD]
headline(f"The rest of this notebook uses: {USE_METHOD} ({len(chunks)} chunks).")

method (sizes in words),chunks,smallest,average,largest
fixed size,83,7,102,120
fixed size with overlap,92,21,106,120
structure-based,161,13,52,260
semantic,57,25,148,200
contextual,161,17,59,267


In [6]:
# @title Turn every chunk into numbers { display-mode: "form" }
embeddings = embedder.encode([chunk.text for chunk in chunks], normalize_embeddings=True)

numbers = "".join(
    f'<span style="display:inline-block;margin:2px;padding:3px 7px;border-radius:5px;background:#dbe8f8;'
    f'font-family:monospace;color:{INK}">{value:+.3f}</span>'
    for value in embeddings[0][:8]
)
display(HTML(model_label()))
headline(f"Each chunk is now a list of {embeddings.shape[1]} numbers ({embeddings.shape[0]} chunks in all).")
display(HTML(card(
    f'<div style="color:{SOFT_INK};margin-bottom:4px">The first 8 numbers of the chunk above:</div>'
    f'{numbers} <span style="color:{GREY}">… and {embeddings.shape[1] - 8} more</span>'
)))

## 2. Store in Chroma
Each record holds the text, its embedding and its source metadata.

In [11]:
# @title Store the chunks in Chroma { display-mode: "form" }
import chromadb

collection = chromadb.EphemeralClient().get_or_create_collection(
    "demo_kb", metadata={"hnsw:space": "cosine"}
)
collection.upsert(
    ids=[chunk.id for chunk in chunks],
    documents=[chunk.text for chunk in chunks],
    embeddings=embeddings.tolist(),
    metadatas=[
        {"title": chunk.title, "department": chunk.department,
         "updated": chunk.updated, "audience": chunk.audience}
        for chunk in chunks
    ],
)

stored = collection.get(ids=[chunks[0].id], include=["metadatas", "documents"])
metadata_rows = "".join(
    f'<tr><td style="padding:4px 16px 4px 0;color:{SOFT_INK}">{html.escape(field)}</td>'
    f'<td style="padding:4px 0;color:{INK}">{html.escape(str(value))}</td></tr>'
    for field, value in stored["metadatas"][0].items()
)
headline(f"{collection.count()} records stored.")
display(HTML(card(
    f'<div style="color:{SOFT_INK};margin-bottom:4px">One record\'s metadata:</div>'
    f'<table style="border-collapse:collapse">{metadata_rows}</table>'
)))

audience,all
department,IT Security
title,Account Lockout
updated,2026-02-03


## 3. Query: keyword vs vector vs hybrid
- **Keyword (BM25)** counts matching words. It's strong on exact terms such as error codes.
- **Vector** compares meaning. It's strong on paraphrases.
- **Hybrid** runs both and combines the two rankings with **reciprocal rank fusion**.

The two searches score in different units, so fusion ignores the scores and uses only the **places**. Each
list gives a chunk points, and the points are added up:

$$\text{score}(\text{chunk}) \;=\; \sum_{\text{each list}} \frac{1}{k + \text{place of the chunk in that list}}
\qquad\text{with } k = 60$$

So 1st place earns $\frac{1}{61}$, 2nd place $\frac{1}{62}$, 3rd place $\frac{1}{63}$, and a chunk missing from a list
earns nothing from it.

**Why 60?** It flattens the gap between places: 1st and 2nd are worth almost the same ($\frac{1}{61}$ vs
$\frac{1}{62}$), so what counts is **being in both lists**. Without it ($k = 0$), 1st place would be worth
twice 2nd place, and a chunk that one search loves would beat one that both searches like. The value
comes from the paper that introduced the method (Cormack, Clarke and Büttcher, 2009): $k = 60$ worked
best in their tests, and the results changed little around it. It's the usual default ever since.

In [12]:
# @title Combine two rankings: reciprocal rank fusion { display-mode: "form" }
QUESTION = "NGL-117 on Wi-Fi"  # @param {type:"string"}

retriever = Retriever(chunks)  # The kit's search tool, built on the chunks above.
TITLES = {d.id: d.title for d in docs}
display(HTML(model_label()))


def reciprocal_rank_fusion(rankings, k=60):
    """Each list gives a chunk 1/(k + its place). Add them up and sort, best first."""
    scores = {}
    for ranking in rankings:
        for place, chunk_id in enumerate(ranking, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + place)
    return sorted(scores, key=scores.get, reverse=True)


keyword = [chunk.id for chunk in retriever.keyword_search(QUESTION, k=5)]
vector = [chunk.id for chunk in retriever.vector_search(QUESTION, k=5)]
hybrid = reciprocal_rank_fusion([keyword, vector])


def place_in(ranking, chunk_id):
    return ranking.index(chunk_id) + 1 if chunk_id in ranking else None


def chunk_name(chunk_id):
    doc_id, number = chunk_id.split("#")
    return f"{TITLES[doc_id]} · chunk {int(number) + 1}"


rows = ""
for final_place, chunk_id in enumerate(hybrid, start=1):
    places = [place_in(keyword, chunk_id), place_in(vector, chunk_id)]
    sums = " + ".join(f"1/(60+{place})" for place in places if place)
    total = sum(1 / (60 + place) for place in places if place)
    shown = ["—" if place is None else str(place) for place in places]
    rows += (
        f'<tr style="{"font-weight:600" if final_place <= 3 else ""}">'
        f'<td style="padding:6px 12px 6px 4px">{html.escape(chunk_name(chunk_id))}</td>'
        f'<td style="padding:6px 12px;text-align:center;color:{ORANGE}">{shown[0]}</td>'
        f'<td style="padding:6px 12px;text-align:center;color:{BLUE}">{shown[1]}</td>'
        f'<td style="padding:6px 12px;font-family:monospace;font-size:12.5px">{sums} = {total:.4f}</td>'
        f'<td style="padding:6px 12px;text-align:center;color:{AQUA}">{final_place}</td></tr>'
    )
header = "".join(
    f'<th style="padding:6px 12px;color:{colour};text-align:{align}">{name}</th>'
    for name, colour, align in (("chunk", SOFT_INK, "left"), ("keyword place", ORANGE, "center"),
                                ("vector place", BLUE, "center"), ("points", SOFT_INK, "left"),
                                ("hybrid place", AQUA, "center"))
)
headline(f'"{html.escape(QUESTION)}": the top 5 of each search, combined')
display(HTML(f'<table style="border-collapse:collapse;background:#fff"><tr>{header}</tr>{rows}</table>'
             f'<div style="color:{SOFT_INK};font-size:12.5px;margin-top:6px">— means not in that top 5, so it adds '
             f'nothing.</div>'))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

chunk,keyword place,vector place,points,hybrid place
VPN Troubleshooting and Error Codes · chunk 2,1,1,1/(60+1) + 1/(60+1) = 0.0328,1
Wi-Fi for Guests · chunk 1,5,3,1/(60+5) + 1/(60+3) = 0.0313,2
Account Lockout · chunk 2,2,—,1/(60+2) = 0.0161,3
VPN Troubleshooting and Error Codes · chunk 3,—,2,1/(60+2) = 0.0161,4
Travel and Roaming · chunk 3,3,—,1/(60+3) = 0.0159,5
IT Help Desk Service Policy · chunk 2,4,—,1/(60+4) = 0.0156,6
VPN Troubleshooting and Error Codes · chunk 1,—,4,1/(60+4) = 0.0156,7
Wi-Fi for Guests · chunk 3,—,5,1/(60+5) = 0.0154,8


### Five test questions
Five questions where we know which document has the answer. Each one is chosen to show where a search
method wins or misses. For each method we show its top three documents: a ✓ means it found the right one.

In [9]:
# @title Five test questions, three ways { display-mode: "form" }
display(HTML(model_label()))  # Vector search turns each question into numbers with the same model.

# Each question: what an employee might type, why we picked it, and the document that has the answer.
QUERIES = [
    ("NGL-809",
     "It tests an exact, rare word: an error code. Keyword search matches it letter for letter; vector "
     "search works from meaning, and a code has little meaning.",
     "vpn-troubleshooting"),
    ("NG-FollowMe",
     "It tests an exact name that only our documents use.",
     "printer-setup"),
    ("how long before my work computer gets swapped for a new one",
     'It tests other words for the same thing: the document says "laptop replacement", not "work '
     'computer swapped", so keyword search has few words to match.',
     "laptop-replacement-policy"),
    ("a visitor at our office wants to get online",
     'It tests other words again: the document says "guest Wi-Fi", not "visitor" or "get online".',
     "guest-wifi"),
    ("my NorthLink connection fails in a hotel abroad",
     'It tests both at once: an exact name ("NorthLink") and a meaning ("hotel abroad" points to the '
     'travel guide).',
     "travel-and-roaming"),
]


def top_docs(hits):
    """The first three different documents among the hits."""
    return list(dict.fromkeys(hit.doc_id for hit in hits))[:3]


def result_column(method, names, wanted):
    """One method's top three, with a ✓ and the place if it found the right document."""
    found = badge(f"✓ found, {['1st', '2nd', '3rd'][names.index(wanted)]}", AQUA) if wanted in names \
        else badge("✗ missed", ORANGE)
    items = "".join(
        f'<div style="{"font-weight:700;color:" + INK if name == wanted else "color:" + SOFT_INK};font-size:13px">'
        f"{number}. {html.escape(TITLES.get(name, name))}</div>"
        for number, name in enumerate(names, start=1)
    )
    return (f'<td style="vertical-align:top;padding:6px 10px 0 0;width:33%">'
            f'<div style="font-weight:600;color:{METHOD_COLOURS[method]};margin-bottom:3px">{method}</div>'
            f"{found}{items}</td>")


for question, why, wanted in QUERIES:
    results = {
        "keyword": top_docs(retriever.keyword_search(question, k=8)),
        "vector": top_docs(retriever.vector_search(question, k=8)),
        "hybrid": top_docs(retriever.hybrid_search(question, k=8)),
    }
    display(HTML(card(
        f'<div style="font-size:15px;font-weight:600">"{html.escape(question)}"</div>'
        f'<div style="color:{SOFT_INK};margin-bottom:6px"><b>The right document:</b> '
        f"{html.escape(TITLES[wanted])}</div>"
        f'<table style="border-collapse:collapse;width:100%"><tr>'
        + "".join(result_column(method, names, wanted) for method, names in results.items())
        + "</tr></table>",
        border=GREY,
    )))

"keyword✓ found, 1st1. VPN Troubleshooting and Error Codes","vector✓ found, 1st1. VPN Troubleshooting and Error Codes2. Printer Setup3. IT Help Desk Service Policy","hybrid✓ found, 1st1. VPN Troubleshooting and Error Codes2. Printer Setup3. IT Help Desk Service Policy"


"keyword✓ found, 1st1. Printer Setup2. IT Onboarding for New Starters","vector✓ found, 1st1. Printer Setup2. Wi-Fi for Guests3. IT Onboarding for New Starters","hybrid✓ found, 1st1. Printer Setup2. IT Onboarding for New Starters3. Wi-Fi for Guests"


"keyword✓ found, 3rd1. Password Policy2. IT Onboarding for New Starters3. Laptop Replacement Policy","vector✓ found, 1st1. Laptop Replacement Policy2. IT Offboarding for Leavers3. IT Onboarding for New Starters","hybrid✓ found, 1st1. Laptop Replacement Policy2. IT Onboarding for New Starters3. Software Request Process"


"keyword✓ found, 1st1. Wi-Fi for Guests2. Mobile Devices3. Shared Drives","vector✓ found, 1st1. Wi-Fi for Guests2. IT Onboarding for New Starters3. VPN Troubleshooting and Error Codes","hybrid✓ found, 1st1. Wi-Fi for Guests2. IT Onboarding for New Starters3. IT Offboarding for Leavers"


"keyword✓ found, 1st1. Travel and Roaming2. VPN Setup Guide3. VPN Troubleshooting and Error Codes","vector✓ found, 3rd1. VPN Troubleshooting and Error Codes2. Teams Meetings3. Travel and Roaming","hybrid✓ found, 1st1. Travel and Roaming2. VPN Troubleshooting and Error Codes3. VPN Setup Guide"


### Search with your own question
Type the question, choose how many results and which method, then run the step. The words of the question
are highlighted in each result.

The HR-only document (salary bands) is hidden from a normal user. Tick **as HR staff** to search the way an
HR employee would; HR-only results get an orange badge.

In [10]:
# @title Search with your own question { display-mode: "form" }
YOUR_QUESTION = "someone keeps trying to log into my account"  # @param {type:"string"}
RESULTS = 3  # @param {type:"slider", min:1, max:5, step:1}
METHOD = "all three"  # @param ["all three", "keyword", "vector", "hybrid"]
AS_HR_STAFF = False  # @param {type:"boolean"}

audiences = ("all", "hr") if AS_HR_STAFF else ("all",)
display(HTML(model_label()))
searches = {
    "keyword": retriever.keyword_search,
    "vector": retriever.vector_search,
    "hybrid": retriever.hybrid_search,
}
chosen = searches if METHOD == "all three" else {METHOD: searches[METHOD]}

show_columns({
    name: "".join(
        hit_card(rank, chunk, YOUR_QUESTION, METHOD_COLOURS[name])
        for rank, chunk in enumerate(search(YOUR_QUESTION, k=RESULTS, audiences=audiences), start=1)
    )
    or card(f'<span style="color:{GREY}">No matching chunk.</span>', border=GREY)
    for name, search in chosen.items()
})

"keyword1. Account Lockout # Account Lockout Your account locks automatically after 5 failed sign-in attempts within 15 minutes. This protects the company from anyone trying to guess passwords. ## What happens next The …account-lockout#02. IT Help Desk Service Policy … tell us what you were trying to do, what happened, and any error code. We will ask for anything else we need.help-desk-policy#33. Password Reset Policy … failed attempts have locked your account, see the Account Lockout guide. ## If you think your password is compromised Reset it straight away and report the event to the help …password-reset-policy#2","vector1. Account Lockout as a possible security incident: 1. Reset your password once the lock lifts, using password.northgate.example. 2. Report it to the help desk and give the time of the lockout. 3. …account-lockout#22. Password Reset Policy … failed attempts have locked your account, see the Account Lockout guide. ## If you think your password is compromised Reset it straight away and report the event to the help …password-reset-policy#23. Account Lockout # Account Lockout Your account locks automatically after 5 failed sign-in attempts within 15 minutes. This protects the company from anyone trying to guess passwords. ## What happens next The …account-lockout#0","hybrid1. Account Lockout # Account Lockout Your account locks automatically after 5 failed sign-in attempts within 15 minutes. This protects the company from anyone trying to guess passwords. ## What happens next The …account-lockout#02. Password Reset Policy … failed attempts have locked your account, see the Account Lockout guide. ## If you think your password is compromised Reset it straight away and report the event to the help …password-reset-policy#23. Email and Calendar … ## Common problems - **Outlook keeps asking for a password:** this often follows a password change. Close Outlook and reopen it, or see Account Lockout if your account has locked. …email-and-calendar#2"
